In [5]:
import random 
import pandas as pd
from haversine import haversine, Unit
import numpy as np

In [13]:
df_test = pd.read_parquet("../data/processed/modified_AIS_data.parquet")
df_test

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,...,Width,Draft,Cargo,TransceiverClass,TimeDiff,SubTrackID,prev_LAT,prev_LON,distance_km,speed_kmph
0,209112000,2024-01-03 13:19:58,48.51548,-126.51201,9.8,98.7,104.0,STALO,IMO9758404,5BQM4,...,32.0,7.5,70.0,A,NaN,0,NaN,NaN,NaN,NaN
1,209112000,2024-01-03 13:32:50,48.51033,-126.45776,10.3,99.7,104.0,STALO,IMO9758404,5BQM4,...,32.0,7.5,70.0,A,772.0,0,48.51548,-126.51201,4.036950,18.825155
2,209112000,2024-01-03 13:34:50,48.50968,-126.44914,10.5,91.4,104.0,STALO,IMO9758404,5BQM4,...,32.0,7.5,70.0,A,120.0,0,48.51033,-126.45776,0.639097,19.172914
3,209112000,2024-01-03 13:40:59,48.50753,-126.42323,10.1,96.0,107.0,STALO,IMO9758404,5BQM4,...,32.0,7.5,70.0,A,369.0,0,48.50968,-126.44914,1.923641,18.767225
4,209112000,2024-01-03 13:42:38,48.50664,-126.41652,9.8,107.4,107.0,STALO,IMO9758404,5BQM4,...,32.0,7.5,70.0,A,99.0,0,48.50753,-126.42323,0.504133,18.332117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462430,677059800,2024-01-08 08:26:19,19.00699,-68.17823,8.2,293.7,294.0,ZULU AJ,IMO8315449,5IM698,...,10.0,3.0,70.0,A,126.0,0,19.00504,-68.17374,0.519466,14.841884
462431,677059800,2024-01-08 08:28:43,19.00889,-68.18343,7.7,280.4,270.0,ZULU AJ,IMO8315449,5IM698,...,10.0,3.0,70.0,A,144.0,0,19.00699,-68.17823,0.586090,14.652246
462432,677059800,2024-01-08 08:30:39,19.00953,-68.18770,7.8,293.4,309.0,ZULU AJ,IMO8315449,5IM698,...,10.0,3.0,70.0,A,116.0,0,19.00889,-68.18343,0.454516,14.105669
462433,677059800,2024-01-08 08:47:23,19.02076,-68.22079,7.8,253.3,265.0,ZULU AJ,IMO8315449,5IM698,...,10.0,3.0,70.0,A,1004.0,0,19.00953,-68.18770,3.696002,13.252595


In [7]:
def compute_distance(row):
    """
    Computes the distance between two geographical points using the Haversine formula

    Args:
        row (pd.Series): A row of the DataFrame containing latitude and longitude of two points

    Returns:
        float: Distance in kilometers
    """
    if pd.notnull(row['prev_LAT']) and pd.notnull(row['prev_LON']):
        coord_prev = (row['prev_LAT'], row['prev_LON'])
        coord_curr = (row['LAT'], row['LON'])
        return haversine(coord_prev, coord_curr, unit=Unit.KILOMETERS)
    else:
        return np.nan

def filter_by_speed(df, min_speed = 1, max_speed = 60):
    """
    Filters the AIS data to include only vessels with a speed between min_speed and max_speed

    Args:
        df (pd.DataFrame): AIS data
        min_speed (float): Minimum speed threshold, defaults to 1
        max_speed (float): Maximum speed threshold, defaults to 60

    """
    # Sort the data and reset index for proper sequential processing
    df = df.sort_values(by=["MMSI", "SubTrackID", "BaseDateTime"]).copy()
    
    # Calculate distance between consecutive points
    df["prev_LAT"] = df.groupby(["MMSI", "SubTrackID"])["LAT"].shift(1)
    df["prev_LON"] = df.groupby(["MMSI", "SubTrackID"])["LON"].shift(1)
    df['distance_km'] = df.apply(compute_distance, axis=1)
    
    # Convert time difference to seconds
    df["TimeDiff"] = df["TimeDiff"].dt.total_seconds()
    
    # Calculate speed
    df["speed_kmph"] = df['distance_km'] / df['TimeDiff'] * 3600
    
    # Calculate average speed for each vessel
    df_avg_speeds = df.groupby(["MMSI", 'SubTrackID'])['speed_kmph'].mean().reset_index()
    
    # Filter vessels based on speed
    valid_tracks = df_avg_speeds[(df_avg_speeds['speed_kmph'] >= min_speed) & (df_avg_speeds['speed_kmph'] <= max_speed)][["MMSI", 'SubTrackID']]
    
    # Merge to get original data
    df_filterd = df.merge(valid_tracks, on=["MMSI", 'SubTrackID'], how='inner')
    
    return df_filterd

In [9]:
df = filter_by_speed(df_test, min_speed=1, max_speed=60)
df

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,...,Width,Draft,Cargo,TransceiverClass,TimeDiff,SubTrackID,prev_LAT,prev_LON,distance_km,speed_kmph
0,209112000,2024-01-03 13:19:58,48.51548,-126.51201,9.8,98.7,104.0,STALO,IMO9758404,5BQM4,...,32.0,7.5,70.0,A,NaN,0,NaN,NaN,NaN,NaN
1,209112000,2024-01-03 13:32:50,48.51033,-126.45776,10.3,99.7,104.0,STALO,IMO9758404,5BQM4,...,32.0,7.5,70.0,A,772.0,0,48.51548,-126.51201,4.036950,18.825155
2,209112000,2024-01-03 13:34:50,48.50968,-126.44914,10.5,91.4,104.0,STALO,IMO9758404,5BQM4,...,32.0,7.5,70.0,A,120.0,0,48.51033,-126.45776,0.639097,19.172914
3,209112000,2024-01-03 13:40:59,48.50753,-126.42323,10.1,96.0,107.0,STALO,IMO9758404,5BQM4,...,32.0,7.5,70.0,A,369.0,0,48.50968,-126.44914,1.923641,18.767225
4,209112000,2024-01-03 13:42:38,48.50664,-126.41652,9.8,107.4,107.0,STALO,IMO9758404,5BQM4,...,32.0,7.5,70.0,A,99.0,0,48.50753,-126.42323,0.504133,18.332117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
550209,677059800,2024-01-08 08:26:19,19.00699,-68.17823,8.2,293.7,294.0,ZULU AJ,IMO8315449,5IM698,...,10.0,3.0,70.0,A,126.0,0,19.00504,-68.17374,0.519466,14.841884
550210,677059800,2024-01-08 08:28:43,19.00889,-68.18343,7.7,280.4,270.0,ZULU AJ,IMO8315449,5IM698,...,10.0,3.0,70.0,A,144.0,0,19.00699,-68.17823,0.586090,14.652246
550211,677059800,2024-01-08 08:30:39,19.00953,-68.18770,7.8,293.4,309.0,ZULU AJ,IMO8315449,5IM698,...,10.0,3.0,70.0,A,116.0,0,19.00889,-68.18343,0.454516,14.105669
550212,677059800,2024-01-08 08:47:23,19.02076,-68.22079,7.8,253.3,265.0,ZULU AJ,IMO8315449,5IM698,...,10.0,3.0,70.0,A,1004.0,0,19.00953,-68.18770,3.696002,13.252595


In [15]:
df_test[['MMSI', 'SubTrackID']].drop_duplicates()


,MMSI,SubTrackID
0,209112000,0
885,209112000,20
1768,209112000,21
1845,209444000,0
2677,209444000,1
...,...,...
460595,636023489,5
461033,636023537,0
461515,636092792,0
461562,636092852,0


In [ ]:
775